view 和 reshape 的区别是 PyTorch 面试和日常开发中的高频考点。
它们的核心区别在于：**对内存连续性的要求**以及**底层执行逻辑**。

以下是为你整理的详细对比表及核心总结：

# 📊 核心对比表

| 对比维度 | `tensor.view()` | `tensor.reshape()` |
| :--- | :--- | :--- |
| 内存连续性要求 | 极其严格：<font color='red'>输入张量必须是内存连续的（`is_contiguous() == True`），否则直接报错。</font> | 宽松：<font color='blue'>不要求内存连续。如果输入不连续，它会在内部自动调用 `contiguous()` 进行内存拷贝。</font> |
| 底层执行逻辑 | **纯粹的元数据修改**：绝不复制数据，只改变步长（strides）等视图属性。 | 智能判断：<font color='red'>先尝试像 `view` 一样零拷贝；如果失败（因为不连续），则退化为 `contiguous() + view`，产生数据拷贝。</font> |
| 返回值性质 | **始终返回原张量的视图（View），修改返回值会直接修改原张量**。 | <font color='red'>如果发生了内存拷贝，返回的是一个独立的新张量（修改它不会影响原张量）；如果没拷贝，则是视图。</font> |
| 性能表现 | <font color='red'>极高</font>：时间复杂度 O(1)，零额外内存开销。 | <font color='red'>视情况而定</font>：内存连续时 O(1)；内存不连续时 O(N)（N为元素总数），有额外的内存分配和拷贝开销。 |
| 典型报错 | `RuntimeError: view size is not compatible with input tensor's size and stride...` | 极少报错，因为它会自动处理不连续的情况。 |

## 💡 核心总结与实战建议
###  一句话概括
  - view 是“**绝对零拷贝**”的硬汉，但脾气倔，内存不连续就直接罢工报错。
  - reshape 是“**智能兼容**”的暖男，内存连续就零拷贝，<font color='red'>不连续就默默帮你拷贝一份再变形</font>。
  
###  工程实战建议
- 追求极致性能且确定内存连续时（例如：刚刚 cat 完、刚创建的张量、没有经过 transpose 的张量）：
  - 👉 优先使用 view()。<font color='red'>这能强制保证你的代码没有发生任何隐式的内存拷贝，防止性能退化。</font>
  
- <font color='red'>写通用代码或不确定内存状态时（例如：经过复杂的 transpose、permute、切片操作后的张量）：</font>
  - 👉 优先使用 reshape()。这能避免程序因为内存不连续而突然崩溃，提高代码的鲁棒性。
- 调试显存/性能瓶颈时：
  - 👉 如果你怀疑某处 reshape 导致了意外的内存拷贝，可以将其替换为 view()。如果报错，说明确实发生了拷贝；如果不报错，说明确实是零拷贝。
  
### 代码验证示例

In [1]:
import torch

x = torch.arange(6).reshape(2, 3)  # 内存连续
y = x.transpose(0, 1)              # 内存不连续

# 1. view 直接报错
# y.view(6)  # RuntimeError!

# 2. reshape 默默成功（内部发生了拷贝）
z = y.reshape(6)  
print(z)  # tensor([0, 3, 1, 4, 2, 5])

# 3. 验证 reshape 在不连续时产生了新张量
print(z.data_ptr() == y.data_ptr())  # False (内存地址不同，说明发生了拷贝)

tensor([0, 3, 1, 4, 2, 5])
False
